In [1]:
# ============================================================================
# make_mixed_tb.py
# Interleaves 100 shots across all 4 frozen modes (25 per mode) and writes
# unified golden test vectors for tb_unified_readout.sv.
# ============================================================================
import os
import numpy as np

MODES = [
    (0, "design_freeze/single",      1, 500),
    (1, "design_freeze/bus5",        5, 500),
    (2, "design_freeze/spin_single", 1, 1600),
    (3, "design_freeze/spin_bus5",   5, 1600),
]
SHOTS_PER_MODE = 25
OUT_DIR = "design_freeze/unified/golden"


def build_mixed_vectors():
    os.makedirs(OUT_DIR, exist_ok=True)
    loaded = {}
    for prof_id, folder, n_q, T in MODES:
        npz = np.load(os.path.join(folder, "golden", "golden_vectors.npz"))
        # Use polynomial golden predictions (matches classifier_poly_unified)
        pred = npz["pred_poly"] if "pred_poly" in npz else npz["pred_lin"]
        loaded[prof_id] = (npz["I"], npz["Q"], pred, n_q, T)

    meta_lines = []   # format: {profile_id[1:0], T[12:0]} -> 16-bit hex
    in_lines = []     # format: {Q[15:0], I[15:0]} -> 32-bit hex
    pred_lines = []   # format: {3'b0, pred[4:0]} -> 8-bit hex

    # Interleave shots across all 4 profiles back-to-back
    for s in range(SHOTS_PER_MODE):
        for prof_id, _, _, _ in MODES:
            I_arr, Q_arr, pred_arr, n_q, T = loaded[prof_id]
            meta_word = ((prof_id & 0x3) << 13) | (T & 0x1FFF)
            meta_lines.append(f"{meta_word:04x}")

            for t in range(T):
                w = ((int(Q_arr[s, t]) & 0xFFFF) << 16) | (int(I_arr[s, t]) & 0xFFFF)
                in_lines.append(f"{w:08x}")

            mask_pred = 0
            for q in range(n_q):
                if int(pred_arr[s, q]) & 1:
                    mask_pred |= (1 << q)
            pred_lines.append(f"{mask_pred:02x}")

    with open(os.path.join(OUT_DIR, "tb_mixed_meta.mem"), "w") as f:
        f.write("\n".join(meta_lines) + "\n")
    with open(os.path.join(OUT_DIR, "tb_mixed_input.mem"), "w") as f:
        f.write("\n".join(in_lines) + "\n")
    with open(os.path.join(OUT_DIR, "tb_mixed_pred.mem"), "w") as f:
        f.write("\n".join(pred_lines) + "\n")

    print(f"[OK] Wrote {len(meta_lines)} interleaved shots ({len(in_lines)} ADC samples) to {OUT_DIR}/")


if __name__ == "__main__":
    build_mixed_vectors()

[OK] Wrote 100 interleaved shots (105000 ADC samples) to design_freeze/unified/golden/
